# Vitrine — Posicionamento de *Datacenters* de Borda (ODCs) em O-RAN

### Baselines multiobjetivo **ao vivo** + heurística **projetada por um LLM**

Este notebook é a **vitrine reprodutível** da implementação, pensada para avaliação
acadêmica. Ele roda **de ponta a ponta no Google Colab** (ou em qualquer máquina),
**sem GPU e sem Ollama** — apenas CPU.

**Estratégia de reprodutibilidade (Abordagem A):**
- Os **baselines** (guloso *k-median* e **NSGA‑II**) são executados **AO VIVO** aqui — são só CPU.
- Os resultados do **LLM** (heurística vencedora, curvas de treino, tabelas de
  generalização, Fase 5b) são **carregados dos artefatos já versionados no repositório**.
  **Não** chamamos o LLM/Ollama ao vivo: a heurística vencedora é **código Python puro**
  e é executada diretamente.

**Portabilidade:** a primeira célula **clona o repositório público** e lê *tudo* de lá
(detectando se já estamos dentro do repo, para não clonar duas vezes). Assim o notebook
funciona igual no Colab e localmente, sem depender de nenhum caminho da máquina do autor.

> Repositório: `https://github.com/jeanzcorreia/open_ran_datacenter_placement`

**Roteiro (espelha as seções do artigo):**
1. Setup — clona o repo e instala dependências.
2. O problema — instâncias reais (Natal, Belo Horizonte) e a formulação multiobjetivo.
3. Baselines **ao vivo** — guloso + NSGA‑II, com as fronteiras de Pareto.
4. A heurística **projetada pelo LLM** — mostra o código e executa a heurística.
5. Resultados — HV treino×teste, **curva de convergência (curva de treino)** e generalização.
6. Fase 5b (diferencial) — objetivo em **linguagem natural**: balanceamento de carga (A vs B).
7. Conclusão — resumo honesto.


## 1. Setup — clonar o repositório e instalar dependências

A célula abaixo é o coração da portabilidade. Ela:
1. **Procura** a raiz do repositório subindo pelos diretórios (marcadores:
   `src/problem/instance.py` + `data/processed/`). Se já estivermos **dentro** do
   repo (uso local), **não clona**.
2. Caso contrário (ex.: Colab), **clona** o repositório público e entra nele.
3. Coloca a raiz do repo no `sys.path`, para podermos `import src...`.

Para testar o caminho *real* do Colab a partir de outro repositório local, defina a
variável de ambiente `VITRINE_REPO_URL` apontando para a origem desejada; por padrão,
usa o GitHub público.

In [ ]:
import os, sys, subprocess, shutil

# Origem do repositório. No Colab usa-se o GitHub público (padrão). Pode ser sobrescrita
# por VITRINE_REPO_URL (ex.: para validar o caminho de clonagem a partir de um espelho local).
REPO_URL = os.environ.get(
    "VITRINE_REPO_URL",
    "https://github.com/jeanzcorreia/open_ran_datacenter_placement.git",
)
REPO_NAME = "open_ran_datacenter_placement"

def find_repo_root(start):
    """Sobe a árvore de diretórios procurando a raiz do repo (por marcadores estáveis)."""
    d = os.path.abspath(start)
    while True:
        if (os.path.isfile(os.path.join(d, "src", "problem", "instance.py"))
                and os.path.isdir(os.path.join(d, "data", "processed"))):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent

# 1) já estamos dentro do repo?
root = find_repo_root(os.getcwd())

# 2) existe um clone anterior nesta pasta? (idempotência ao reexecutar)
if root is None and os.path.isdir(REPO_NAME):
    root = find_repo_root(os.path.abspath(REPO_NAME))
    if root is None:                      # clone anterior incompleto -> remove
        shutil.rmtree(REPO_NAME, ignore_errors=True)

# 3) senão, clona
if root is None:
    print(f"Clonando {REPO_URL} ...")
    depth = ["--depth", "1"] if REPO_URL.startswith("http") else []  # shallow só p/ http(s)
    subprocess.run(["git", "clone", *depth, REPO_URL, REPO_NAME], check=True)
    root = find_repo_root(os.path.abspath(REPO_NAME))

assert root, "Não foi possível localizar a raiz do repositório após o clone."
os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

REPO_ROOT = root
print("REPO_ROOT =", REPO_ROOT)
print("Conteúdo de data/processed/:", sorted(os.listdir("data/processed"))[:12])


Dependências científicas. No Colab, `numpy/pandas/scikit-learn/matplotlib` já vêm
instalados; ainda assim instalamos tudo (incl. **`pymoo`**, que fornece NSGA‑II e o
cálculo de *hypervolume*) para funcionar em qualquer máquina. Fixamos `pymoo` na série
0.6.x (API estável usada pelo projeto).

In [ ]:
import sys, subprocess
# instala no MESMO interpretador do kernel (funciona no Colab e localmente)
pkgs = ["numpy", "pandas", "scikit-learn", "matplotlib", "pymoo>=0.6.1,<0.7"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib", "inline")   # figuras embutidas no notebook
except Exception:
    pass
import pymoo, sklearn
print("numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| pymoo", pymoo.__version__,
      "| matplotlib", matplotlib.__version__)

# figuras: renderizadas inline E salvas em disco (evidência de execução)
FIGDIR = os.path.join(REPO_ROOT, "figuras_vitrine")
os.makedirs(FIGDIR, exist_ok=True)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 110
import warnings; warnings.filterwarnings("ignore")   # silencia avisos ruidosos do pymoo/sklearn
print("Figuras serão salvas em:", FIGDIR)


## 2. O problema — instâncias reais e a formulação multiobjetivo

Cada **cidade** é uma instância construída a partir de dados públicos de estações da Anatel
(`data/processed/<Cidade>.csv`). No **modo justo** (o usado no artigo):

- **Candidatos a ODC** = todos os **sites** únicos da cidade (dedupe por `cell_site_id`).
- **Clientes** = todas as linhas do CSV (uma O‑RU por linha), cada uma com uma demanda de
  CPU derivada da largura de banda (designação de emissão ITU).

**Objetivos (ambos MINIMIZADOS):**
- **f1 = nº de ODCs ativos** (custo de infraestrutura);
- **f2 = distância média de *fronthaul*** (km, Haversine cliente→ODC mais próximo).

**Restrições (viável ⇔ ambas nulas):**
- capacidade por ODC ≤ **1000** núcleos de CPU;
- distância cliente→ODC ≤ **11 km**.

Carregamos os parâmetros direto do artefato `report_data.json` (fonte única da verdade) e
duas instâncias — **Natal** (pequena, usada ao vivo) e **Belo Horizonte** (grande).

In [ ]:
import json
from src.problem.instance import load_instance_sites

# Parâmetros do problema — lidos do artefato versionado (mesma fonte da rodada oficial).
report = json.load(open("Results/phase5a/report_data.json", encoding="utf-8"))
PROB = report["problem"]
MD, MC, CPU = PROB["max_distance"], PROB["max_capacity"], PROB["cpu_per_100mhz"]
print(f"Parâmetros do cenário: max_distance={MD} km | max_capacity={MC} cores | cpu_per_100mhz={CPU}")

def fix_txt(s):
    """Repara acentos com dupla codificação (UTF-8 lido como latin-1) que aparecem em alguns
    rótulos gravados no artefato; deixa strings já corretas intactas."""
    try:
        return s.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

split = report["split"]
disp = {k: fix_txt(v) for k, v in split.get("display_names", {}).items()}
print("\nSeparação treino/validação (definida ANTES de treinar):")
print("  TREINO  (6):", [disp.get(k, k) for k in split["train"]])
print("  VALIDAÇÃO held-out (4):", [disp.get(k, k) for k in split["test"]])

# Duas instâncias reais
rows = []
insts = {}
for city in ["Natal", "BeloHorizonte"]:
    inst = load_instance_sites(f"data/processed/{city}.csv", cpu_per_100mhz=CPU)
    insts[city] = inst
    rows.append(dict(cidade=disp.get(city, city), n_sites=inst.n_unique_sites,
                     n_clientes=inst.n_clients, demanda_CPU_total=inst.total_cpu_demand))
tbl = pd.DataFrame(rows).set_index("cidade")
print()
tbl


## 3. Baselines **ao vivo** (só CPU)

Executamos **de verdade**, aqui e agora, dois baselines sobre **Natal**:

- **Guloso (*k‑median* construtivo)** — determinístico: a cada passo adiciona o site que mais
  reduz a distância média, gerando uma solução por `k` (`src/optimizers/fair.py:GreedyFair`).
- **NSGA‑II** (pymoo) — algoritmo evolutivo multiobjetivo de referência.

> ⏱️ Para a demonstração ao vivo ser rápida em CPU, o NSGA‑II roda com uma **configuração
> reduzida** (`pop`/`gerações` menores). Isto é apenas ilustrativo — os números com a
> **configuração completa** (pop=300, 60 gerações, 5 seeds) estão na **Seção 5**, carregados
> dos artefatos. O guloso é determinístico e roda na configuração plena.

Ambos os métodos são reavaliados no **problema verdadeiro** (`FairODCProblem`) e reduzidos ao
conjunto **viável e não‑dominado** — comparação justa no mesmo espaço de objetivos.

In [ ]:
import time
from src.optimizers.fair import FAIR_OPTIMIZERS
from src.optimizers.llm.reevo import instance_hv   # HV normalizado por instância (métrica interna, ilustrativa)

CITY = "Natal"
inst = insts[CITY]
dist_max = float(inst.distances.max())
print(f"Cidade ao vivo: {disp.get(CITY, CITY)} — {inst.n_unique_sites} sites, {inst.n_clients} clientes\n")

# --- Guloso (determinístico, configuração completa) ---
t0 = time.time()
ps_greedy = FAIR_OPTIMIZERS["greedy"](MD, MC).solve(inst, budget=0, seed=0)
t_greedy = time.time() - t0
print(f"Guloso: {int(ps_greedy.feasible.sum())} pontos viáveis não-dominados  ({t_greedy:.1f}s)")

# --- NSGA-II (configuração REDUZIDA para a demo ao vivo) ---
NSGA_POP, NSGA_GEN = 80, 40
t0 = time.time()
ps_nsga = FAIR_OPTIMIZERS["nsga2"](MD, MC, pop_size=NSGA_POP).solve(inst, budget=NSGA_GEN, seed=1)
t_nsga = time.time() - t0
print(f"NSGA-II (pop={NSGA_POP}, gen={NSGA_GEN}): {int(ps_nsga.feasible.sum())} pontos viáveis  ({t_nsga:.1f}s)")


In [ ]:
def front_xy(ps):
    """(f1, f2) dos pontos VIÁVEIS não-dominados, ordenados por f1 (para desenhar a fronteira)."""
    F = np.atleast_2d(ps.F)[np.asarray(ps.feasible, dtype=bool)]
    F = F[np.argsort(F[:, 0])]
    return F[:, 0], F[:, 1]

fig, ax = plt.subplots()
for ps, lbl, style in [(ps_greedy, "Guloso (k-median)", "o-"),
                       (ps_nsga, f"NSGA-II (pop={NSGA_POP}, {NSGA_GEN} ger.)", "s--")]:
    x, y = front_xy(ps)
    ax.plot(x, y, style, ms=4, lw=1.3, alpha=0.85, label=lbl)
ax.set_xlabel("f1 = nº de ODCs ativos  (↓ melhor)")
ax.set_ylabel("f2 = distância média de fronthaul [km]  (↓ melhor)")
ax.set_title(f"Fronteiras de Pareto ao vivo — {disp.get(CITY, CITY)}")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIGDIR, "fig3_baselines_natal.png")); plt.show()

# HV interno (ilustrativo, normalizado por [n_sites, dist_max]) — maior é melhor
for ps, lbl in [(ps_greedy, "Guloso"), (ps_nsga, "NSGA-II(reduzido)")]:
    F = np.atleast_2d(ps.F)[np.asarray(ps.feasible, dtype=bool)]
    print(f"  HV interno {lbl:18s}: {instance_hv(F, inst.n_unique_sites, dist_max):.4f}")


## 4. A heurística **projetada pelo LLM**

Na Fase 5a, um laço evolutivo guiado por LLM (**ReEvo**, multi‑cidade) *projeta o código* de
uma heurística construtiva `place_odcs(instance, n_active)`. A **vencedora** foi salva no repo
como código Python puro — reproduzimo‑la aqui **sem nenhuma chamada ao LLM**.

> **Sobre segurança:** no *pipeline* de treino, cada heurística gerada pelo LLM (código não
> confiável) roda num **sandbox isolado** — validação por **AST**, *builtins* restritos e
> *timeout* real (`src/optimizers/llm/heuristic_runtime.py`). Como aqui a vencedora já é
> **código fixo, revisado e versionado**, nós a executamos **diretamente** (mais portável no
> Colab) — reutilizando o **mesmo modelo de objetivos** do projeto (`FairODCProblem` +
> fronteira viável não‑dominada), de modo que a fronteira produzida é idêntica à do
> *pipeline* oficial.

In [ ]:
def read_text(path):
    """Lê texto tentando UTF-8 e caindo para latin-1 (alguns artefatos .py foram gravados em cp1252)."""
    b = open(path, "rb").read()
    try:
        return b.decode("utf-8")
    except UnicodeDecodeError:
        return b.decode("latin1")

winner_code = read_text("Results/phase5a/winning_heuristic_multicity.py")
print("=" * 78)
print("HEURÍSTICA VENCEDORA (Fase 5a, ReEvo multi-cidade) — código versionado no repo:")
print("=" * 78)
print(winner_code)


In [ ]:
# Executa a heurística CONFIÁVEL diretamente (sem subprocesso), reutilizando o modelo de
# objetivos do projeto, para gerar a fronteira dela em uma cidade.
from src.problem.odc_problem import FairODCProblem
from src.optimizers.base import feasible_nd_front
from src.optimizers.llm.heuristic_runtime import HeuristicInstance, _normalize_selection
from src.optimizers.llm.reevo_multicity import build_capped_sweep

def heuristic_front(code, inst, md=MD, mc=MC, max_points=200):
    """Aplica place_odcs num sweep de n_active e devolve (F, viável) da fronteira não-dominada."""
    # No pipeline de treino o código do LLM passa por validação AST antes de rodar isolado;
    # aqui a vencedora é código CONFIÁVEL e versionado, então a executamos diretamente.
    ns = {}
    exec(compile(code, "<winner>", "exec"), ns)          # código confiável -> define place_odcs
    place = ns["place_odcs"]
    fair = FairODCProblem(inst, md, mc)
    hinst = HeuristicInstance.from_instance(inst, md, mc)
    sweep = build_capped_sweep(inst, mc, max_points=max_points)  # nº de ODCs de mín_viável..n_sites
    Xs = []
    for n in sweep:
        idx = _normalize_selection(place(hinst, int(n)), hinst.n_sites)  # normalizador do próprio repo
        x = np.zeros(hinst.n_sites); x[idx] = 1.0
        Xs.append(x)
    X = np.array(Xs)
    F, G = fair.evaluate_population(X)                    # objetivos/viabilidade VERDADEIROS
    Xf, Ff, feas = feasible_nd_front(X, F, G)
    return Ff, feas

t0 = time.time()
Ff_h, feas_h = heuristic_front(winner_code, inst)
print(f"Heurística do LLM em {disp.get(CITY, CITY)}: {int(feas_h.sum())} pontos viáveis não-dominados "
      f"({time.time()-t0:.1f}s)")


In [ ]:
# Comparação no MESMO gráfico: guloso vs NSGA-II vs heurística do LLM (Natal, ao vivo)
def xy(F, feas):
    F = np.atleast_2d(F)[np.asarray(feas, dtype=bool)]
    F = F[np.argsort(F[:, 0])]
    return F[:, 0], F[:, 1]

fig, ax = plt.subplots()
gx, gy = front_xy(ps_greedy);  ax.plot(gx, gy, "o-", ms=4, lw=1.3, alpha=.85, label="Guloso (k-median)")
nx, ny = front_xy(ps_nsga);    ax.plot(nx, ny, "s--", ms=4, lw=1.3, alpha=.85, label=f"NSGA-II (reduzido)")
hx, hy = xy(Ff_h, feas_h);     ax.plot(hx, hy, "^-", ms=4, lw=1.6, alpha=.95, label="Heurística do LLM")
ax.set_xlabel("f1 = nº de ODCs ativos  (↓ melhor)")
ax.set_ylabel("f2 = distância média de fronthaul [km]  (↓ melhor)")
ax.set_title(f"Baselines vs heurística do LLM — {disp.get(CITY, CITY)} (ao vivo)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIGDIR, "fig4_llm_vs_baselines_natal.png")); plt.show()

hv_h = instance_hv(np.atleast_2d(Ff_h)[np.asarray(feas_h, dtype=bool)], inst.n_unique_sites, dist_max)
print(f"  HV interno Heurística do LLM: {hv_h:.4f}")
print("  (na config completa e com a fronteira-referência por cidade, os números rigorosos estão na Seção 5)")


## 5. Resultados — HV treino×validação, curva de treino e generalização *(obrigatório)*

Aqui carregamos os resultados **oficiais** (configuração completa, salvos em
`Results/phase5a/report_data.json`) e reproduzimos as figuras do artigo.

**Separação treino/validação (nossa validação held‑out):**
- **Treino = 6 cidades** — Manaus, Natal, Belo Horizonte, Goiânia, João Pessoa, Campo Grande.
- **Validação = 4 cidades held‑out** — Curitiba, Recife, Florianópolis, Vitória.

A heurística é **projetada usando apenas as 6 cidades de treino** e depois aplicada, sem
re‑evoluir, às 4 cidades de validação — exatamente para medir **generalização**.

O **hypervolume (HV)** desta seção é normalizado pela **fronteira‑referência por cidade**
(união dos não‑dominados de todos os métodos); **maior é melhor**.

In [ ]:
# ---- HV por cidade (média sobre seeds), com marcação treino/validação ----
cm = report["city_metrics"]
train_set = set(split["train"])
methods = ["reevo", "zero_shot", "greedy", "nsga2", "nsga3", "moead", "random"]
mlabel = {"reevo": "Heur. LLM (ReEvo)", "zero_shot": "Heur. LLM (zero-shot)", "greedy": "Guloso",
          "nsga2": "NSGA-II", "nsga3": "NSGA-III", "moead": "MOEA/D", "random": "Aleatório"}

rows = []
for city in split["train"] + split["test"]:
    if city not in cm:
        continue
    r = {"cidade": disp.get(city, city), "grupo": "treino" if city in train_set else "validação"}
    for m in methods:
        r[mlabel[m]] = round(cm[city][m]["HV"][0], 3) if m in cm[city] else np.nan
    rows.append(r)
hv_table = pd.DataFrame(rows).set_index(["grupo", "cidade"])
print("HV por cidade (↑ melhor) — média sobre seeds:")
hv_table


In [ ]:
# ---- Tabela de GENERALIZAÇÃO: HV médio por grupo (treino 6 vs validação 4) ----
gen = report["generalization"]
grows = []
for m in methods:
    if m not in gen:
        continue
    g = gen[m]
    grows.append(dict(metodo=mlabel[m],
                      HV_treino_media=round(g["train_mean"], 3),
                      HV_treino_maximin=round(g["train_maximin"], 3),
                      HV_valid_media=round(g["test_mean"], 3),
                      HV_valid_maximin=round(g["test_maximin"], 3),
                      gap=round(g["gap"], 3)))
gen_table = pd.DataFrame(grows).set_index("metodo")
print("Generalização — HV médio por grupo (gap = treino - validação; negativo = melhor na validação):")
gen_table


In [ ]:
# ---- HV treino vs validação: gráfico de barras (heurística do LLM em destaque) ----
fig, ax = plt.subplots(figsize=(9, 5))
labels = [mlabel[m] for m in methods if m in gen]
xtr = [gen[m]["train_mean"] for m in methods if m in gen]
xte = [gen[m]["test_mean"] for m in methods if m in gen]
xpos = np.arange(len(labels)); w = 0.38
ax.bar(xpos - w/2, xtr, w, label="treino (6 cidades)")
ax.bar(xpos + w/2, xte, w, label="validação held-out (4)")
ax.set_xticks(xpos); ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("HV médio (↑ melhor)")
ax.set_title("Generalização — HV médio treino vs validação held-out")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(os.path.join(FIGDIR, "fig5a_generalizacao_hv.png")); plt.show()


### Curva de convergência = **curva de treino** do método

A curva abaixo é a **curva de treino** do nosso método: a evolução do **HV de treino**
(agregado sobre as 6 cidades) **por geração** do laço evolutivo guiado por LLM. É o análogo,
para este método, de uma curva de *loss/accuracy* por época.

> **Leitura honesta:** nesta configuração (LLM **local de 7B**, população 8, 6 gerações), a
> curva é **praticamente plana** — a heurística vencedora já apareceu na **população inicial**
> (semente forte + inicializações do LLM), e as gerações seguintes **não a superaram**
> (`best_origin = "seed"`). Ou seja, o laço evolutivo **não agregou** ganho de HV *nesta*
> configuração; discutimos isso na conclusão. A curva de treino é reportada mesmo assim, por
> exigência metodológica e por transparência.

*(HV aqui é a métrica interna de treino, normalizada por instância — não confundir com o HV
por fronteira‑referência das tabelas acima.)*

In [ ]:
# ---- Curva de treino: HV por geração (representativa + por seed) ----
tm = report["train_meta"]
mean_curve = tm["mean_hv_curve"]
gens = list(range(len(mean_curve)))

fig, ax = plt.subplots()
# curvas por seed (transparentes) + representativa (grossa)
psm = report.get("per_seed_train_meta", {})
for s, d in psm.items():
    c = d.get("mean_hv_curve")
    if c:
        ax.plot(range(len(c)), c, "-", lw=1, alpha=0.4, label=f"seed {s}")
ax.plot(gens, mean_curve, "o-", lw=2.5, color="black", label="representativa (melhor seed)")
ax.set_xlabel("geração")
ax.set_ylabel("HV de treino (interno, ↑ melhor)")
ax.set_title("Curva de treino — HV de treino por geração (ReEvo multi-cidade)")
_allv = list(mean_curve) + [v for d in psm.values() for v in (d.get("mean_hv_curve") or [])]
_lo, _hi = min(_allv), max(_allv)
_pad = max(0.02, (_hi - _lo) * 0.25)
ax.set_ylim(_lo - _pad, _hi + _pad)
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIGDIR, "fig5b_curva_treino.png")); plt.show()

print(f"HV de treino (repres.): início={mean_curve[0]:.4f} -> fim={mean_curve[-1]:.4f} "
      f"| best_origin={tm.get('best_origin')} | offspring avaliados={tm.get('total_offspring_evaluated')}")


## 6. Fase 5b (diferencial) — objetivo em **linguagem natural**: balanceamento

O diferencial do trabalho: descrever um **novo objetivo em linguagem natural** e deixar o LLM
**projetar a heurística** que o atende. Aqui o objetivo é *"equilibrar a carga entre os ODCs"*.

Comparamos duas configurações, aplicadas às mesmas instâncias:
- **A (sem balanceamento):** heurística evoluída visando só distância/custo.
- **B (com balanceamento):** heurística evoluída a partir do objetivo textual de **equilibrar a carga**.

A métrica é o **desbalanceamento** de carga entre ODCs (menor é melhor). Resultados carregados
de `Results/phase5b/` (versionados).

In [ ]:
# ---- Tabela A vs B (desbalanceamento por cidade) ----
rep5b = json.load(open("Results/phase5b/rep_table.json", encoding="utf-8"))
b_rows = []
for city in ["Natal", "BeloHorizonte"]:
    a = rep5b[f"A (sem balanceamento)|{city}"]
    b = rep5b[f"B (com balanceamento)|{city}"]
    red = (b["mean_imbalance"] - a["mean_imbalance"]) / a["mean_imbalance"] * 100.0
    b_rows.append(dict(cidade=disp.get(city, city),
                       desbal_A=round(a["mean_imbalance"], 3),
                       desbal_B=round(b["mean_imbalance"], 3),
                       reducao_pct=round(red, 1),
                       jain_A=round(a["mean_jain"], 3), jain_B=round(b["mean_jain"], 3)))
bal_table = pd.DataFrame(b_rows).set_index("cidade")
print("Balanceamento de carga — A (sem) vs B (com), objetivo dado em LINGUAGEM NATURAL:")
print("(desbalanceamento: menor é melhor; Jain: maior = mais justo)")
bal_table


In [ ]:
# ---- Gráfico de barras A vs B + código da balanceadora ----
fig, ax = plt.subplots()
cities = list(bal_table.index)
xpos = np.arange(len(cities)); w = 0.38
ax.bar(xpos - w/2, bal_table["desbal_A"], w, label="A (sem balanceamento)")
ax.bar(xpos + w/2, bal_table["desbal_B"], w, label="B (com balanceamento, NL)")
for i, city in enumerate(cities):
    ax.annotate(f"{bal_table['reducao_pct'][city]:+.0f}%",
                (xpos[i] + w/2, bal_table["desbal_B"][city]),
                ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_xticks(xpos); ax.set_xticklabels(cities)
ax.set_ylabel("desbalanceamento de carga (↓ melhor)")
ax.set_title("Fase 5b — objetivo em linguagem natural reduz o desbalanceamento")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(os.path.join(FIGDIR, "fig6_balanceamento_AvsB.png")); plt.show()

# agregado (todas as cidades) do artefato de análise
ra = json.load(open("Results/phase5b/rebalance_analysis.json", encoding="utf-8"))
print(f"Agregado (melhor balanceador): imbalance A={ra['A_baseline']['best_bal']['imbN']:.3f} "
      f"-> B={ra['B_balanced']['best_bal']['imbN']:.3f}")

print("\n" + "=" * 78)
print("BALANCEADORA B (projetada pelo LLM a partir do objetivo textual):")
print("=" * 78)
print(read_text("Results/phase5b/best_balancer_B_balanced.py"))


## 7. Conclusão — resumo honesto

- **Competitiva:** a heurística projetada pelo LLM fica **no páreo com o NSGA‑II** em HV
  (ver Seção 5), sendo *muito* mais barata de avaliar (código construtivo, sem população
  evolutiva por cidade).
- **Generaliza:** treinada em **6 cidades** e aplicada a **4 cidades held‑out**, mantém o HV
  (o *gap* treino→validação é **negativo** — desempenho na validação ≥ treino), e roda
  **viável e sem falhas** em todas as 10 cidades.
- **Não bate o guloso:** o **guloso *k‑median*** tem o **maior HV** neste problema — é um
  baseline forte, e não o superamos. Relatamos isso abertamente.
- **O laço evolutivo não agregou nesta configuração:** com o **LLM local de 7B**, a vencedora
  veio da **população inicial**; a curva de treino é plana (Seção 5).
- **Diferencial (Fase 5b):** descrever o objetivo em **linguagem natural** ("equilibrar a
  carga") e deixar o LLM projetar a heurística **funciona** — o desbalanceamento cai
  **≈29% (Natal)** e **≈47% (Belo Horizonte)** em relação à versão sem balanceamento.

Todas as figuras foram salvas em `figuras_vitrine/`.

In [ ]:
print("Figuras geradas em figuras_vitrine/:")
for f in sorted(os.listdir(FIGDIR)):
    print("  -", f)
print("\nNotebook executado de ponta a ponta com sucesso. ✔")
